In [1]:
from pathlib import Path
import pandas as pd

path = Path("data/Transcripts/AAPL/2016-Apr-26-AAPL.txt")

# Full transcript as one string
text = path.read_text(encoding="utf-8", errors="replace")

# Optional: one row per line for easier preprocessing
transcript_df = pd.DataFrame({"line": text.splitlines()})
print(transcript_df.shape)
transcript_df

(490, 1)


,line
0,
1,
2,Thomson Reuters StreetEvents Event Transcript
3,E D I T E D V E R S I O N
4,
...,...
485,CONFERENCE CALL ITSELF AND THE APPLICABLE COMP...
486,MAKING ANY INVESTMENT OR OTHER DECISIONS.
487,----------------------------------------------...
488,Copyright 2019 Thomson Reuters. All Rights Res...


In [2]:
import json
import re
with open("test_transcript.json", "r") as f:
    data = json.load(f)

text = "Q2 2016 Apple Inc Earnings Call"
match = re.search(r'(Q[1-4])\s(\d{4})\s(.+?)\sEarnings Call', text)

if match:
    quarter = match.group(1)[1]  # Q2
    year = match.group(2)     # 2016
    company = match.group(3)  # Apple Inc
    data["Company"] = company
    data["Year"] = year
    data["Quarter"] = quarter


with open("test_transcript.json", "w") as f:
    json.dump(data, f, indent=4)

In [4]:
name_line_mask = transcript_df['line'].str.match(r'^\s*\*')

# Keep only the participant text after optional whitespace + leading '*'.
participant_names = (
    transcript_df.loc[name_line_mask, 'line']
    .str.replace(r'^\s*\*\s*', '', regex=True)
    .str.strip()
    .reset_index(drop=True)
)
participant_roles = transcript_df.loc[name_line_mask.shift(1, fill_value=False), 'line'].reset_index(drop=True)

pattern = r'^\s*(?P<company>.+?)\s*-\s*(?P<role>.+?)\s*$'
participant_tuples = [
    (name, match.group('company').strip(), match.group('role').strip())
    for name, role_line in zip(participant_names, participant_roles)
    for match in [re.match(pattern, role_line)]
    if match
]

# Normalize company strings so 'Apple Inc' and 'Apple Inc.' match.
def normalize_company_name(value: str) -> str:
    return re.sub(r'\.', '', value or '').strip().casefold()

company_key = normalize_company_name(company)
corporate = [entry for entry in participant_tuples if normalize_company_name(entry[1]) == company_key]
analysts = [entry for entry in participant_tuples if normalize_company_name(entry[1]) != company_key]

data['Corporate'] = corporate
data['Analysts'] = analysts

with open('test_transcript.json', 'w') as f:
    json.dump(data, f, indent=4)

print('Corporate:', corporate)
print('Analysts:', analysts)


Corporate: [('Luca Maestri', 'Apple Inc.', 'CFO'), ('Tim Cook', 'Apple Inc.', 'CEO'), ('Nancy Paxton', 'Apple Inc.', 'Senior Director of IR')]
Analysts: [('Katy Huberty', 'Morgan Stanley', 'Analyst'), ('Gene Munster', 'Piper Jaffray & Co.', 'Analyst'), ('Rod Hall', 'JPMorgan', 'Analyst'), ('Shannon Cross', 'Cross Research', 'Analyst'), ('Toni Sacconaghi', 'Bernstein', 'Analyst'), ('Simona Jankowski', 'Goldman Sachs', 'Analyst'), ('Steve Milunovich', 'UBS', 'Analyst')]


In [60]:
def get_rows_between_speakers(SPEAKER_1: int, SPEAKER_2: int) -> pd.DataFrame:
    """Return transcript rows strictly between two speaker rows from rows_with_bracket_digit.

    SPEAKER_1 and SPEAKER_2 are 1-indexed row positions in rows_with_bracket_digit.
    """
    if not isinstance(SPEAKER_1, int) or not isinstance(SPEAKER_2, int):
        raise TypeError('SPEAKER_1 and SPEAKER_2 must be integers.')

    bracket_digit_mask = transcript_df["line"].str.contains(r"\[\d+\]", regex=True, na=False)
    speaker_rows = transcript_df.loc[bracket_digit_mask].copy()
    total_speakers = len(speaker_rows)

    if total_speakers < 2:
        return transcript_df.iloc[0:0].copy()

    if SPEAKER_1 < 1 or SPEAKER_2 < 1 or SPEAKER_1 > total_speakers or SPEAKER_2 > total_speakers:
        raise IndexError(f'SPEAKER_1 and SPEAKER_2 must be between 1 and {total_speakers}.')

    if abs(SPEAKER_1 - SPEAKER_2) != 1:
        raise ValueError('SPEAKER_1 and SPEAKER_2 must be adjacent speaker numbers.')

    sectioned_speaker_rows = speaker_rows.reset_index(names='transcript_row_index')
    sectioned_speaker_rows['transcript_row_index'] = sectioned_speaker_rows['transcript_row_index'] + 1

    start_pos, end_pos = sorted((SPEAKER_1 - 1, SPEAKER_2 - 1))
    row_index1 = sectioned_speaker_rows.loc[start_pos, 'transcript_row_index']
    row_index2 = sectioned_speaker_rows.loc[end_pos, 'transcript_row_index']

    between_rows = transcript_df.iloc[row_index1 : row_index2 - 1, :].copy()
    valid_text_mask = (
        between_rows['line'].str.strip().ne('')
        & between_rows['line'].str.contains(r'[A-Za-z]', regex=True, na=False)
    )
    return between_rows.loc[valid_text_mask].reset_index(drop=True)

rows_between_first_second = get_rows_between_speakers(SPEAKER_1=22, SPEAKER_2=23)
rows_between_first_second


,line
0,"Katy, in the short term, let me just..."
1,But the other multiplier in that equation is o...
2,"From an India point of view, if you look at In..."
3,"Unlike the US as an example, where the carrier..."
4,We've been working in India now for a couple o...


In [42]:
bracket_digit_mask = transcript_df["line"].str.contains(r"\[\d+\]", regex=True, na=False)
speaker_rows = transcript_df.loc[bracket_digit_mask].copy()

# Extract numeric id from patterns like [1], [12], etc.
speaker_rows['speaker_num'] = (
    speaker_rows['line']
    .str.extract(r'\[(\d+)\]')[0]
    .astype(int)
)

# Start a new section whenever the speaker number drops (e.g., 5 -> 1).
speaker_rows['section_id'] = (speaker_rows['speaker_num'].diff().lt(0)).cumsum() + 1

# Optional helper numbering inside each section.
speaker_rows['position_in_section'] = speaker_rows.groupby('section_id').cumcount() + 1

sectioned_speaker_rows = speaker_rows.reset_index(names='transcript_row_index')
sectioned_speaker_rows['transcript_row_index'] = sectioned_speaker_rows['transcript_row_index'] + 1
sections = [
    section_df.reset_index(drop=True)
    for _, section_df in sectioned_speaker_rows.groupby('section_id', sort=True)
]

sectioned_speaker_rows

,transcript_row_index,line,speaker_num,section_id,position_in_section
0,43,Operator [1],1,1,1
1,49,"Nancy Paxton, Apple Inc. - Senior Director of...",2,1,2
2,58,"Tim Cook, Apple Inc. - CEO [3]",3,1,3
3,88,"Luca Maestri, Apple Inc. - CFO [4]",4,1,4
4,120,"Nancy Paxton, Apple Inc. - Senior Director of...",5,1,5
5,132,Operator [1],1,2,1
6,139,"Simona Jankowski, Goldman Sachs - Analyst [2]",2,2,2
7,145,"Luca Maestri, Apple Inc. - CFO [3]",3,2,3
8,151,"Tim Cook, Apple Inc. - CEO [4]",4,2,4
9,161,"Simona Jankowski, Goldman Sachs - Analyst [5]",5,2,5


In [51]:
s1 = 2
s2 = 3
row_index1, row_index2 = sectioned_speaker_rows['transcript_row_index'][s1], sectioned_speaker_rows['transcript_row_index'][s2]
print(row_index1, row_index2)

58 88


In [52]:
transcript_df.iloc[row_index1:row_index2-1, :]

,line
58,----------------------------------------------...
59,
60,"Thanks, Nancy, and good afternoon, e..."
61,"Revenue for the quarter was $50.6 billion, whi..."
62,We saw continued currency weakness in the vast...
63,"Despite challenges, there were a number of enc..."
64,"We sold 51.2 million iPhones in the quarter, c..."
65,As we look at each of these three sources of i...
66,"Second, we continued to see a very high level ..."
67,"And third, with only 42% smart phone penetrati..."
